# A3 · Standard errors of the coefficients: closed formula versus bootstrap

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 40 minutes · **extended path**

## Objective

Compute the uncertainty of the OLS coefficients in two ways, with the analytic formula and with the pairs bootstrap, and
show that the two results coincide only if the errors are homoscedastic.

## Dataset

Synthetic, generated by the function `generate(heteroscedastic)`: 150 rows, 3 Gaussian features, true coefficients
`[1, 2, -1, 0.5]` (intercept included). With `heteroscedastic=False` the noise has constant standard deviation (2.0);
with `True` it grows with the absolute value of the first feature.

## Educational purpose

You already know the theory: `Var(β̂) = σ² (AᵀA)⁻¹`. Here you implement it, compare it with an estimate that makes no assumptions about the
variance of the errors, and see in a concrete case what happens to the t tests and the confidence intervals when
homoscedasticity does not hold.

## Task

1. Write `standard_errors_formula(X, y)`: it returns an array with the standard errors of `[intercept, coef...]` computed as the square root of the diagonal of `σ̂² (AᵀA)⁻¹`, with `σ̂² = SSR / (n − p − 1)`.
2. Write `standard_errors_bootstrap(X, y, B=2000, seed=0)`: it resamples B times the **rows** (x, y pairs) with replacement, re-estimates the OLS with `np.linalg.lstsq` and returns `(se, coef_boot)`, where `se` is the standard deviation (`ddof=1`) of each coefficient and `coef_boot` the B×4 matrix of the estimates.
3. On the homoscedastic dataset compute `ratio_homo = se_boot / se_formula` (one value per coefficient).
4. On the heteroscedastic dataset compute `ratio_het` the same way. Assign to `underestimates_x1` `True` if for the coefficient of the **first feature** (index 1) the ratio exceeds 1.3, that is if the formula underestimates the uncertainty.
5. With the homoscedastic bootstrap build the 95% confidence interval (2.5 and 97.5 percentiles) of each coefficient and save in `ci_width` the width of the intervals.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

TRUE_BETA = np.array([1.0, 2.0, -1.0, 0.5])


def generate(heteroscedastic, seed=1, n=150):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, 3))
    sigma = 2.0 * (1.5 * np.abs(X[:, 0]) + 0.1) if heteroscedastic else 2.0
    y = TRUE_BETA[0] + X @ TRUE_BETA[1:] + rng.normal(size=n) * sigma
    return X, y


X_homo, y_homo = generate(False)
X_het, y_het = generate(True)

## Your work

In [ ]:
def standard_errors_formula(X, y):
    A = np.column_stack([np.ones(len(X)), X])
    n, k = A.shape                       # k = number of coefficients, intercept included
    ...


def standard_errors_bootstrap(X, y, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    A = np.column_stack([np.ones(len(X)), X])
    coef_boot = np.empty((B, A.shape[1]))
    for b in range(B):
        ...
    return coef_boot.std(axis=0, ddof=1), coef_boot

In [ ]:
se_formula_homo = standard_errors_formula(X_homo, y_homo)
se_boot_homo, coef_boot_homo = standard_errors_bootstrap(X_homo, y_homo)
ratio_homo = ...

In [ ]:
se_formula_het = standard_errors_formula(X_het, y_het)
se_boot_het, _ = standard_errors_bootstrap(X_het, y_het)
ratio_het = ...
underestimates_x1 = ...

In [ ]:
ci = np.percentile(coef_boot_homo, [2.5, 97.5], axis=0)
ci_width = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Bootstrap: `idx = rng.integers(0, n, n)` draws the indices with replacement; then `coef_boot[b] = np.linalg.lstsq(A[idx], y[idx], rcond=None)[0]`. Formula: `sigma2 = ssr / (n - k)` and `np.sqrt(np.diag(sigma2 * np.linalg.inv(A.T @ A)))`.'):
    A = np.column_stack([np.ones(len(X_homo)), X_homo])
    e = y_homo - A @ np.linalg.lstsq(A, y_homo, rcond=None)[0]
    expected = np.sqrt(np.diag(e @ e / (len(y_homo) - 4) * np.linalg.inv(A.T @ A)))
    check(close(se_formula_homo, expected), "the analytic standard errors are correct",
          "standard_errors_formula does not match sqrt(diag(sigma2 * inv(A^T A))): check the degrees of freedom n-p-1")
    check(coef_boot_homo.shape == (2000, 4), "the bootstrap returns B x 4 estimates", "coef_boot must have shape (B, 4)")
    check(np.all((ratio_homo > 0.85) & (ratio_homo < 1.2)),
          "with homoscedastic errors formula and bootstrap agree (ratios between 0.85 and 1.2)",
          "on the homoscedastic dataset the two methods should give similar standard errors")
    check(bool(underestimates_x1) and ratio_het[1] > 1.3,
          "with heteroscedastic errors the formula underestimates the uncertainty of the first feature",
          "on the heteroscedastic dataset the ratio for the first feature should exceed 1.3")
    check(close(ci_width, 3.92 * se_boot_homo, rtol=0.15),
          "the 95% intervals are about 3.92 standard errors wide",
          "ci_width should be ci[1] - ci[0], that is about 2 * 1.96 * se")
done()

---
The commented solution is in `solutions/A_from_scratch/A3_standard_errors_formula_and_bootstrap.ipynb`. Open it after you pass the check, or after you have really tried.